# 世界坐标系 → 相机坐标系 → 像素坐标系完整投影

给定世界坐标系中的三维点、相机外参和相机内参，依次完成世界坐标系 → 相机坐标系 → 像素坐标系的正向投影。

## 1. 坐标系与投影链路

常见的 OpenCV 相机坐标系约定为：

- $X_c$ 轴向右；
- $Y_c$ 轴向下；
- $Z_c$ 轴朝向相机前方。

像素坐标系通常约定原点位于图像左上角，$u$ 轴向右，$v$ 轴向下。

完整链路为：

```text
世界坐标系中的 3D 点
        ↓ 外参变换
相机坐标系中的 3D 点
        ↓ 内参投影
像素坐标系中的 2D 点
```


## 2. 相机外参 $R$、$t$

相机外参描述世界坐标系和相机坐标系之间的相对位置与姿态。它由旋转矩阵 $\mathbf{R}$ 和平移向量 $\mathbf{t}$ 组成。

在本节中，外参采用"世界坐标系 → 相机坐标系"的约定：

$$
\mathbf{P}_c=\mathbf{R}_{cw}\mathbf{P}_w+\mathbf{t}_{cw}
$$

其中：

- $\mathbf{P}_w$：点在世界坐标系中的坐标；
- $\mathbf{P}_c$：同一个点在相机坐标系中的坐标；
- $\mathbf{R}_{cw}$：把世界坐标方向转换为相机坐标方向的旋转；
- $\mathbf{t}_{cw}$：世界坐标到相机坐标变换中的平移项。

齐次形式为：

$$
\begin{bmatrix}\mathbf{P}_c\\1\end{bmatrix}=\mathbf{T}_{cw}\begin{bmatrix}\mathbf{P}_w\\1\end{bmatrix},\qquad
\mathbf{T}_{cw}=\begin{bmatrix}\mathbf{R}_{cw}&\mathbf{t}_{cw}\\0&1\end{bmatrix}
$$

这里的 $\mathbf{t}_{cw}$ 不一定是"相机在世界坐标中的位置"。如果相机光心在世界坐标系中的位置是 $\mathbf{C}_w$，则：

$$
\mathbf{t}_{cw}=-\mathbf{R}_{cw}\mathbf{C}_w,\qquad
\mathbf{C}_w=-\mathbf{R}_{cw}^{T}\mathbf{t}_{cw}
$$

如果已知的是相机坐标系到世界坐标系的位姿 $\mathbf{T}_{wc}$，则需要取逆变换：

$$
\mathbf{P}_w=\mathbf{R}_{wc}\mathbf{P}_c+\mathbf{t}_{wc}
$$

其逆变换为：

$$
\mathbf{P}_c=\mathbf{R}_{wc}^{T}(\mathbf{P}_w-\mathbf{t}_{wc})
$$

因此，使用外参前必须先确认矩阵描述的是哪个方向；不同资料对下标命名可能不同。


## 3. 正向投影步骤

给定世界坐标系中的点 $\mathbf{P}_w$，先使用相机外参得到相机坐标：

$$
\mathbf{P}_c=\mathbf{R}_{cw}\mathbf{P}_w+\mathbf{t}_{cw}
$$

再使用相机内参将 $\mathbf{P}_c$ 投影到像素平面。完整链路为：

$$
\mathbf{P}_w \xrightarrow{\mathbf{R},\mathbf{t}} \mathbf{P}_c \xrightarrow{\mathbf{K}\text{ 与透视除法}} (u,v)
$$


## 4. 相机坐标系 → 像素坐标系

设相机坐标系中的三维点为：

$$
\mathbf{P}_c=\begin{bmatrix}X_c\\Y_c\\Z_c\end{bmatrix}
$$

透视投影先得到归一化平面坐标：

$$
x=\frac{X_c}{Z_c},\qquad y=\frac{Y_c}{Z_c}
$$

再通过相机内参得到像素坐标：

$$
u=f_x\frac{X_c}{Z_c}+c_x,\qquad
v=f_y\frac{Y_c}{Z_c}+c_y
$$

内参矩阵为：

$$
\mathbf{K}=\begin{bmatrix}
f_x&0&c_x\\
0&f_y&c_y\\
0&0&1
\end{bmatrix}
$$

齐次形式为：

$$
s\begin{bmatrix}u\\v\\1\end{bmatrix}=\mathbf{K}\begin{bmatrix}X_c\\Y_c\\Z_c\end{bmatrix}
$$

其中，

$$
\boxed{s = 齐次尺度因子}
$$

而在这里：

$$
\boxed{s = Z_c}
$$

## 5. 有效投影

只有位于相机前方的点才有有效投影：

$$
Z_c>0
$$

此外，还需要检查 $(u,v)$ 是否落在图像范围内。

In [ ]:
import numpy as np

def world_to_camera(P_w, R, t):
    """将世界坐标系中的点变换到相机坐标系。"""
    P_w = np.asarray(P_w, dtype=float)
    R = np.asarray(R, dtype=float)
    t = np.asarray(t, dtype=float)
    return R @ P_w + t


def camera_to_pixel(P_c, K):
    """将相机坐标系中的点投影到像素坐标。"""
    P_c = np.asarray(P_c, dtype=float)
    K = np.asarray(K, dtype=float)

    X_c, Y_c, Z_c = P_c
    if Z_c <= 0:
        return None

    p = K @ P_c
    return p[:2] / p[2]


def project_world_to_pixel(P_w, R, t, K, image_size=None):
    """完成世界坐标系到像素坐标系的完整投影。"""
    P_c = world_to_camera(P_w, R, t)
    pixel = camera_to_pixel(P_c, K)

    if pixel is None or image_size is None:
        return P_c, pixel

    width, height = image_size
    u, v = pixel
    in_image = 0 <= u < width and 0 <= v < height
    return P_c, pixel if in_image else None

In [ ]:
# 相机内参和外参示例
K = np.array([
    [1000.0, 0.0, 960.0],
    [0.0, 1000.0, 540.0],
    [0.0, 0.0, 1.0]
])
R = np.eye(3)
t = np.zeros(3)
image_size = (1920, 1080)

points_world = [
    np.array([0.0, 0.0, 5.0]),
    np.array([1.0, 0.5, 5.0]),
    np.array([-1.0, 0.5, 5.0]),
    np.array([1.0, -0.5, 5.0]),
    np.array([1.0, 0.5, 10.0]),
]

for i, P_w in enumerate(points_world, start=1):
    P_c, pixel = project_world_to_pixel(P_w, R, t, K, image_size)
    print(f"点 {i}: 世界坐标={P_w}, 相机坐标={P_c}, 像素坐标={pixel}")